# CollectiveLens in Google Colab

Run the app in your browser. Nothing is installed on your computer.

**Before you start (optional, for written answers):** add your Anthropic API key as a Colab secret.
Click the key icon in the left bar, choose *Add new secret*, name it `ANTHROPIC_API_KEY`, paste the key as the value
and switch on *Notebook access*. A key comes from console.anthropic.com (it is separate from a Claude subscription).
Without a key everything works except the written answers and summaries.

**Then:** choose *Runtime > Run all*, or run the four steps below one after the other. The first run takes a few minutes,
because the multilingual search model (about 2 GB) is downloaded. At the end, open the link printed under
*Running on public URL* in a new browser tab.

Good to know:

* The link is public and temporary. Anyone who has it can use the app, and written answers are paid with your key
  (limited to 40 calls a day by default). Do not share it, and stop the last cell when you are done.
* Colab forgets everything when the session ends, so next time you run the cells again.
* The demo library is fictional. Do not upload confidential documents to a notebook unless your data policy allows it.
* The app runs as a separate program, so you do not need to restart the session after the installation.


In [ ]:
# Step 1. Get the code
import io, os, pathlib, shutil, subprocess, zipfile

REPO = "https://github.com/fabianamaraffa/CollectiveLens.git"
BRANCH = "main"            # use "v2" if version 2 is still on its own branch
APP_DIR = pathlib.Path("/content/CollectiveLens")


def has_code(path):
    return all((path / name).exists() for name in
               ("collectivelens/__init__.py", "app.py", "data/sample/library/documents.json"))


if not has_code(APP_DIR):
    shutil.rmtree(APP_DIR, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(APP_DIR)],
                   capture_output=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})

if not has_code(APP_DIR):
    # Version 2 is not on GitHub yet (or the repository is private): upload the zip instead.
    print("Choose collectivelens-v2.zip on your computer:")
    from google.colab import files

    uploaded = files.upload()
    zips = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zips:
        raise SystemExit("No .zip file received. Run this cell again and choose collectivelens-v2.zip.")
    unpacked = pathlib.Path("/content/_unpacked")
    shutil.rmtree(unpacked, ignore_errors=True)
    with zipfile.ZipFile(io.BytesIO(uploaded[zips[0]])) as z:
        z.extractall(unpacked)
    marker = next(unpacked.rglob("collectivelens/__init__.py"), None)
    if marker is None:
        raise SystemExit("This zip does not contain the CollectiveLens code.")
    shutil.rmtree(APP_DIR, ignore_errors=True)
    shutil.move(str(marker.parent.parent), str(APP_DIR))

version = (APP_DIR / "collectivelens" / "__init__.py").read_text().split("__version__")[1].split('"')[1]
print(f"CollectiveLens {version} is in {APP_DIR}")


In [ ]:
# Step 2. Install what the app needs (Colab already has PyTorch, so this is quicker than on a laptop)
import subprocess, sys

os.chdir(APP_DIR)
result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
                        capture_output=True, text=True)
if result.returncode != 0:
    print(result.stdout[-3000:], result.stderr[-3000:])
    raise SystemExit("The installation failed. The message above says why.")
print("Python", sys.version.split()[0], "- installation done.")


In [ ]:
# Step 3. Your Anthropic key (optional). It stays in this session and is never printed.
import getpass

key = os.environ.get("ANTHROPIC_API_KEY", "")
if not key:
    try:
        from google.colab import userdata

        key = userdata.get("ANTHROPIC_API_KEY") or ""
    except Exception:
        key = ""
if not key:
    key = getpass.getpass("Anthropic API key (press Enter to skip): ").strip()
if key:
    os.environ["ANTHROPIC_API_KEY"] = key
print("Written answers: " + ("on" if key else "off (search, topics and charts still work)"))


In [ ]:
# Step 4. Start the app. Keep this cell running; stop it (square button) when you are done.
QUICK_LOOK = False   # True skips the 2 GB search model and matches by shared words only (faster, less capable)

# The app runs as its own program. This keeps it apart from the packages Colab already loaded,
# and the share link is what makes it reachable from your browser.
env = {**os.environ, "COLLECTIVELENS_SHARE": "1", "COLLECTIVELENS_OPEN_BROWSER": "0", "PYTHONUNBUFFERED": "1"}
if QUICK_LOOK:
    env["COLLECTIVELENS_EMBEDDER"] = "lite"

app = subprocess.Popen([sys.executable, "app.py"], cwd=APP_DIR, env=env,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in app.stdout:
        print(line, end="")
        if "public URL" in line:
            print(">>> Open the link above in a new browser tab. Do not share it.\n")
except KeyboardInterrupt:
    pass
finally:
    app.terminate()
    print("The app has stopped.")


When the app is open, the line at the top should say **Search by meaning** (and **Claude on** if you gave a key).
If it says keyword matching instead, the message next to it explains why.

Things to try with the demo library:

* Search `finanziamenti europei per piccole associazioni`. The English document *Simpler funding for small
  organisations* should come first, because the model matches meaning across languages.
* In *Ask the library*: `What do the documents recommend to make exchange programmes fairer?`
  The answer should cite passages as [1], [2], with the passages listed below it.
* Open *Insights* to see the topics and the map of the library.
